In [ ]:
from google.colab import drive

drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
import os
os.chdir('/content/gdrive/MyDrive/College/NLP')

In [ ]:
import sys
sys.path.append('/content/gdrive/MyDrive/College/NLP')

import geval_prompts

In [ ]:
from google.colab import userdata
import os


os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [82]:
from omegaconf import OmegaConf

cfg = OmegaConf.create({
    "model_name": "gpt-o3-mini",
    "data_model": "aya-expanse-8b",

    #Data response
    "data_path": "/content/gdrive/MyDrive/College/NLP/outputs/{data_model}/{strategy}/{culture}_response.json",

    #Save the ouput
    "output_dir": "/content/gdrive/MyDrive/College/NLP/outputs/evaluations",

    "culture": "Spanish",
    "strategy": "redditor",
})

In [ ]:
import json
import os
from tqdm import trange
import torch
import re

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from openai import OpenAI

from geval_prompts import *
import numpy as np

def load_dataset(cfg):
    cfg.data_path = cfg.data_path.format(data_model=cfg.data_model, strategy=cfg.strategy, culture=cfg.culture)
    print(f"Loading data file: {cfg.data_path}")
    dataset = []
    with open(cfg.data_path, 'r') as f:
        for l in f.readlines():
            dataset.append(json.loads(l))
    print(f"Size of the data {len(dataset)}")
    return dataset

class GEvaluator:
    def __init__(self, cfg):
        self.cfg = cfg

    def load_model(self):
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )

        _model = AutoModelForCausalLM.from_pretrained(
            self.cfg.model_path,
            cache_dir="/content/hf_cache",
            local_files_only=True,
            torch_dtype=torch.float16,
            device_map="auto",
            quantization_config=quantization_config
        )
        self.model = torch.compile(_model, mode='max-autotune')
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.cfg.model_path,
            padding_side="left",
        )

    def load_gpt_model(self):
        self.model = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

    def get_evaluation_prompt(self, metric: str, post: str, response: str):
        metric_def = definition(metric)
        eval_steps = steps(metric)
        return EVAL_PROMPT.format(metric=metric, metric_def=metric_def,
                                  eval_steps=eval_steps,
                                  post=post, response=response)

    def generate_responses(self, prompts):
        scores = []
        for prompt in prompts:
            try:
                response = self.model.chat.completions.create(
                    model="o3-mini-2025-01-31",
                    messages=[{"role": "user", "content": prompt}]
                )
                content = response.choices[0].message.content

                score = self.parse_results(content)
                scores.append(score)
            except Exception as e:
                print(f"Error API: {e}")
                scores.append(0)
        return scores

        inputs = self.tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to("cuda")

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                return_dict_in_generate=True,
                output_scores=True,
                do_sample=False,
                temperature=1.0,
                top_p=1.0,
                top_k=0
            )
        torch.cuda.empty_cache()
        return outputs

    def get_all_emotion_metric_prompts(self, post, response):
        return [self.get_evaluation_prompt(m, post, response) for m in EMO_METRICS]

    def get_all_cultural_metric_prompts(self, post, response):
        return [self.get_evaluation_prompt(m, post, response) for m in CULT_METRICS]

    def get_all_quality_metric_prompts(self, post, response):
        return [self.get_evaluation_prompt(m, post, response) for m in OVERALL_METRICS]

    def parse_results(self, results, scores_only=False):
        scores = []

        if "gpt" in self.cfg.model_name.lower():
            output_strings = results
        else:
            output_strings = self.tokenizer.batch_decode(results.sequences, skip_special_tokens=True)

        for i in output_strings:
            match = re.search(r'\b([1-5])\b', i)
            if match:
                scores.append(int(match.group(1)))
            else:
                print(f"Error parsing score from: {i[:50]}...")
                scores.append(0)


        probs = [0] * len(scores)
        if not scores_only and "gpt" not in self.cfg.model_name.lower():
            try:
                token_idx = torch.stack([i[-1] for i in results.sequences])
                batch_indices = torch.arange(token_idx.size(0), device=token_idx.device)
                probs = results.scores[0].softmax(dim=-1)
                probs = probs[batch_indices, token_idx].cpu().detach().tolist()
            except Exception as e:
                print(f"Failed to calculate the probability: {e}")

        return scores, probs

    def evaluate_responses(self, dataset, scores_only=False):
        output_dir = os.path.join(self.cfg.output_dir, self.cfg.model_name, self.cfg.data_model, self.cfg.strategy)

        os.makedirs(output_dir, exist_ok=True)

        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
            print(f"Directory '{output_dir}' created.")
        else:
            print(f"Directory '{output_dir}' already exists.")

        output_file = open(os.path.join(self.cfg.output_dir, self.cfg.model_name, self.cfg.data_model, self.cfg.strategy, f"{self.cfg.culture}_evaluation.json"), 'a')
        for idx in trange(len(dataset)):
            row = dataset[idx]
            emo_prompts = self.get_all_emotion_metric_prompts(row["post"], row["response"])
            cult_prompts = self.get_all_cultural_metric_prompts(row["post"], row["response"])
            quality_prompts = self.get_all_quality_metric_prompts(row["post"], row["response"])

            emo_results = self.generate_responses(emo_prompts)
            cult_results = self.generate_responses(cult_prompts)
            quality_results = self.generate_responses(quality_prompts)

            if "gpt" in self.cfg.model_name:
                emo_scores = emo_results
                cult_scores = cult_results
                quality_scores = quality_results
                emo_probs = [1]*len(emo_scores)
                cult_probs = [1]*len(cult_scores)
                quality_probs = [1]*len(quality_scores)
                norm_score = [score * prob for score, prob in zip(emo_scores+cult_scores+quality_scores, emo_probs+cult_probs+quality_probs)]
            else:
                emo_scores, emo_probs = self.parse_results(emo_results, scores_only)
                cult_scores, cult_probs = self.parse_results(cult_results, scores_only)
                quality_scores, quality_probs = self.parse_results(quality_results, scores_only)
                norm_score = [score * prob for score, prob in zip(emo_scores+cult_scores+quality_scores, emo_probs+cult_probs+quality_probs)]

            data_to_save = {
                "post_id": row["post_id"],
                "post": row["post"],
                "response": row["response"],
                "emo_scores": emo_scores,
                "emo_probs": emo_probs,
                "cult_scores": cult_scores,
                "cult_probs": cult_probs,
                "quality_scores": quality_scores,
                "quality_probs": quality_probs,
                "normalized_score": np.mean(norm_score),
            }
            output_file.write(json.dumps(data_to_save)+"\n")


In [83]:
dataset = load_dataset(cfg)

Loading data file: /content/gdrive/MyDrive/College/NLP/outputs/aya-expanse-8b/redditor/Spanish_response.json
Size of the data 20


In [84]:

evaluator = GEvaluator(cfg)

if "gpt" in cfg.model_name:
    evaluator.load_gpt_model()
else:
    evaluator.load_model()

print("Model loaded")
evaluator.evaluate_responses(dataset)

Model loaded
Directory '/content/gdrive/MyDrive/College/NLP/outputs/evaluations/gpt-o3-mini/aya-expanse-8b/redditor' already exists.


100%|██████████| 20/20 [05:42<00:00, 17.11s/it]


### Cleaning the data

In [85]:
formatted_input_path = os.path.join(
    cfg.output_dir,
    cfg.model_name,
    cfg.data_model,
    cfg.strategy,
    f"{cfg.culture}_evaluation.json"
)

output_folder = os.path.join(cfg.output_dir, cfg.model_name, cfg.data_model, cfg.strategy)
output_file_path = os.path.join(output_folder, f"{cfg.culture}_cleaned.csv")

In [86]:
formatted_input_path

'/content/gdrive/MyDrive/College/NLP/outputs/evaluations/gpt-o3-mini/aya-expanse-8b/redditor/Spanish_evaluation.json'

In [40]:
import pandas as pd

def extract_clean_scores(data):

    text_data = str(data)
    found_scores = re.findall(r'\b[1-5]\b', text_data)
    return [int(s) for s in found_scores]

def process_cleaning():
    print(f"Reading file: {formatted_input_path}")

    if not os.path.exists(formatted_input_path):
        print(f"Error: File not found at {formatted_input_path}")
        return

    # Load JSONL Data
    df = pd.read_json(formatted_input_path, lines=True)

    # Clean Score Columns
    score_cols = ['emo_scores', 'cult_scores', 'quality_scores']

    for col in score_cols:
        if col in df.columns:
            # Flatten lists and remove 0s
            df[col] = df[col].apply(extract_clean_scores)
            # Calculate individual metric means (ignoring empty lists)
            df[f'{col}_mean'] = df[col].apply(lambda x: np.mean(x) if x else np.nan)

    # Calculate Final Normalized Score
    # Aggregates all valid numbers from all three categories for a true average
    def calculate_final_average(row):
        combined_list = row['emo_scores'] + row['cult_scores'] + row['quality_scores']
        return round(np.mean(combined_list), 2) if combined_list else 0.0

    df['normalized_score'] = df.apply(calculate_final_average, axis=1)

    # 6. Save Cleaned Data
    os.makedirs(output_folder, exist_ok=True)
    df.to_csv(output_file_path, index=False)

    print("-" * 30)
    print(f"SUCCESS: Data cleaned for {cfg.culture} ({cfg.strategy})")
    print(f"Cleaned file saved to: {output_file_path}")
    print(df[['post_id', 'normalized_score']].head())



In [87]:
# Run the cleaning process
process_cleaning()

Reading file: /content/gdrive/MyDrive/College/NLP/outputs/evaluations/gpt-o3-mini/aya-expanse-8b/redditor/Spanish_evaluation.json
------------------------------
SUCCESS: Data cleaned for Spanish (redditor)
Cleaned file saved to: /content/gdrive/MyDrive/College/NLP/outputs/evaluations/gpt-o3-mini/aya-expanse-8b/redditor/Spanish_cleaned.csv
   post_id  normalized_score
0  1qtde3h              3.86
1  1c34kgo              4.14
2  1lx71oq              4.14
3  1rd6439              3.71
4  1kjr4fc              4.00
